# 🧪 PT-W2-D3 概念实验：Lifecycle + Event 抽取——状态机 + Effect

> 配套阅读：`PT-W2-D3-Lifecycle-Event抽取.md`
> effect-registry.yaml 的 5 类冻结效应，构成 AI Agent 推理骨架。

## 第 1 格：5 类冻结 Effect 类型

In [ ]:
EFFECT_REGISTRY = {
    "state-transition-effect": {"target": "*", "owner": "各域自管"},
    "occupancy-effect":          {"target": "Resource Unit", "owner": "Lease/Occupancy"},
    "financial-effect":         {"target": "Bill/AR", "owner": "Billing & AR"},
    "lead-conversion-effect":   {"target": "Lead", "owner": "招商"},
    "maintenance-effect":       {"target": "OperationTask", "owner": "运营"},
}

print("effect-registry.yaml v1.0（5 类冻结）：")
for k, v in EFFECT_REGISTRY.items():
    print(f"  {k:<25} → target: {v['target']:<14} owner: {v['owner']}")

## 第 2 格：资源七态状态机 + 合同状态机

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class TransitionDecl:
    from_state: str
    to_state: str
    guard: str
    event: str
    effects: tuple

# 资源七态 canonical
RESOURCE_STATES = ["planned", "created", "available", "reserved", "in-use", "suspended", "retired"]
RESOURCE_TRANSITIONS = {
    ("available", "reserved"):  TransitionDecl("available", "reserved", "预定生效", "ResourceReserved", ("occupancy-effect",)),
    ("reserved", "available"):  TransitionDecl("reserved", "available", "预定到期释放", "ReservationExpired", ("occupancy-effect",)),
    ("available", "in-use"):    TransitionDecl("available", "in-use", "合同生效", "ContractActivated", ("occupancy-effect",)),
    ("in-use", "available"):    TransitionDecl("in-use", "available", "退场完成", "OccupancyReleased", ("occupancy-effect",)),
    ("available", "suspended"):  TransitionDecl("available", "suspended", "暂停经营", "ResourceSuspended", ("state-transition-effect",)),
    ("suspended", "available"):  TransitionDecl("suspended", "available", "恢复经营", "ResourceResumed", ("state-transition-effect",)),
}

# 合同状态机
CONTRACT_STATES = ["draft", "signed", "active", "expiring", "expired", "terminated", "voided"]
CONTRACT_TRANSITIONS = {
    ("draft", "signed"):     TransitionDecl("draft", "signed", "双方签署", "ContractSigned", ()),
    ("signed", "active"):    TransitionDecl("signed", "active", "生效条件满足", "ContractActivated", ("occupancy-effect", "financial-effect")),
    ("active", "expiring"):  TransitionDecl("active", "expiring", "到期预警", "ContractExpiring", ()),
    ("expiring", "expired"):  TransitionDecl("expiring", "expired", "到期日到达", "ContractExpired", ("occupancy-effect",)),
    ("active", "terminated"): TransitionDecl("active", "terminated", "终止审批通过 ∧ 退场完成", "ContractTerminated", ("occupancy-effect", "financial-effect")),
    ("signed", "voided"):     TransitionDecl("signed", "voided", "作废审批通过", "ContractVoided", ("occupancy-effect",)),
}

print("资源迁移:")
for (f, t), d in RESOURCE_TRANSITIONS.items():
    print(f"  {f:>10} → {t:<10} Event: {d.event}  Effects: {d.effects}")
print(f"\n合同迁移:")
for (f, t), d in CONTRACT_TRANSITIONS.items():
    print(f"  {f:>10} → {t:<12} Event: {d.event}  Effects: {d.effects}")

## 第 3 格：迁移引擎 + Event Log

In [ ]:
@dataclass
class Instance:
    obj_type: str
    obj_id: str
    state: str

def try_transition(inst, to_state, decls, facts, event_log):
    decl = decls.get((inst.state, to_state))
    if decl is None:
        return f"❌ 非法迁移 {inst.state}→{to_state}"
    for cond in decl.guard.split(" ∧ "):
        if not facts.get(cond.strip(), False):
            return f"⏸️ 守卫未满足: {cond}"
    inst.state = to_state
    event_log.append((inst.obj_id, decl.event))
    return f"✅ {decl.event}"

resource = Instance("Resource Unit", "A101", "available")
contract = Instance("Contract", "CT001", "signed")
event_log = []

facts = {"合同生效": True}
print("合同 signed→active:", try_transition(contract, "active", CONTRACT_TRANSITIONS, facts, event_log))
print("Event Log:", event_log)
print(f"  Contract state = {contract.state}")

## 第 4 格：可视化——状态机图

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc")
font_name = font_manager.FontProperties(fname="/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc").get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 资源七态
for i, s in enumerate(RESOURCE_STATES):
    ax1.text(0.5, i, s, ha="center", va="center", fontsize=9,
             bbox=dict(boxstyle="round", fc="#E3F2FD" if s != "in-use" else "#FFCDD2", ec="#333"))
for (f, t) in RESOURCE_TRANSITIONS:
    fi, ti = RESOURCE_STATES.index(f), RESOURCE_STATES.index(t)
    ax1.annotate("", xy=(0.3, ti), xytext=(0.3, fi),
                arrowprops=dict(arrowstyle="->", color="#1565C0", lw=1.2))
ax1.set_xlim(-0.5, 1.5); ax1.set_ylim(-0.5, len(RESOURCE_STATES)-0.5)
ax1.set_title("资源七态状态机")
ax1.axis("off")

# 合同状态机
for i, s in enumerate(CONTRACT_STATES):
    ax2.text(0.5, i, s, ha="center", va="center", fontsize=9,
             bbox=dict(boxstyle="round", fc="#E8F5E9" if s != "active" else "#FFE0B2", ec="#333"))
for (f, t) in CONTRACT_TRANSITIONS:
    fi, ti = CONTRACT_STATES.index(f), CONTRACT_STATES.index(t)
    ax2.annotate("", xy=(0.3, ti), xytext=(0.3, fi),
                arrowprops=dict(arrowstyle="->", color="#2E7D32", lw=1.2))
ax2.set_xlim(-0.5, 1.5); ax2.set_ylim(-0.5, len(CONTRACT_STATES)-0.5)
ax2.set_title("合同状态机")
ax2.axis("off")

plt.tight_layout()
plt.savefig("/root/learning-notebooks/第10周/d3_lifecycle.png", dpi=100)
plt.show()
print("状态机图已绘制")

## 第 5 格：Agent 推理链——"A101 为什么不能出租？"

In [ ]:
print("""Agent 推理路径：
1. Entity: Resource A101 → 状态=in-use
2. Lifecycle: in-use → available 需迁移 OccupancyReleased
3. Relationship: A101 关联 Contract CT001 (active)
4. Rule: active 合同未终止前，资源不可出租
5. Event: 需 ContractTerminated 事件 → 触发 occupancy-effect 释放

结论：铺位被合同占用，需先终止合同。Agent 引用的是声明式规则，不是代码 if-else。
""")